In [3]:
import os
import sys
import time
import sqlite3
import pandas as pd
import undetected_chromedriver as uc
import gspread
import base64
import json
import re
import math
from PIL import Image  # ← 後段のWebP変換で必要
from datetime import datetime, timedelta
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from google.oauth2.service_account import Credentials

# === 初期設定 ===
start_time = time.time()

# === pic用のフォルダに使う時間 ===
today = (datetime.now() - timedelta(days=1)).strftime("%Y%m%d")

# 現在の日時
now = datetime.now()

# SKU 用の日付（フォーマットを変えたり今日にしたりも可）
sku_date = datetime.now().strftime("%Y%m%d")  # 例: 当日の日付で SKU 採番

# jupyter-pyどちらもいけるようにパス設定共通
try:
    base_dir = os.path.dirname(__file__)
except NameError:
    base_dir = os.getcwd()

sys.path.append(os.path.abspath(os.path.join(base_dir, "..")))
# ★ TODAY_MODE を追加インポート
from utils.config import PROJECT_DIR, GSHEET_NAME, SHEET_NAME, TODAY_MODE  # 3つまとめて import + TODAY_MODE

# パス定義共通
if os.name == 'nt':
    user_base = os.path.join(os.environ["USERPROFILE"], "myenv310", PROJECT_DIR)
else:
    user_base = os.path.join(os.path.expanduser("~"), "myenv310", PROJECT_DIR)

# dbベース定義
db_path = os.path.join(user_base, "db", "output.db")
columns_file = os.path.join(user_base, "db", "columns_with_type.txt")
table_name = "result_table"

base_path = os.path.join(user_base, "pic", today)  # PNG保存先
os.makedirs(base_path, exist_ok=True)

# === WebP出力先 ===
webp_dir = os.path.join(user_base, "output", "pic", today, "webp")
os.makedirs(webp_dir, exist_ok=True)

# ---- クッキー保存先（ここが今回の主目的）----
credentials_dir = os.path.join(user_base, "credentials")
os.makedirs(credentials_dir, exist_ok=True)
COOKIE_FILE = os.path.join(credentials_dir, "cookies.json")

# === csv出力先===
csv_path = os.path.join(base_path, "output.csv")
excel_path = os.path.join(base_path, "output.xlsx")

# ========= スプレッドシート認証 =========
json_file_path = os.path.join(user_base, "credentials", "gspread-test-402206-686fe226262e.json")
if not os.path.exists(json_file_path):
    raise FileNotFoundError(f"[ERROR] JSONファイルが見つかりません: {json_file_path}")

scope = ['https://www.googleapis.com/auth/spreadsheets', 'https://www.googleapis.com/auth/drive']
credentials = Credentials.from_service_account_file(json_file_path, scopes=scope)
gs = gspread.authorize(credentials)

spreadsheet = gs.open(GSHEET_NAME)
worksheet_slot = spreadsheet.worksheet(SHEET_NAME)
print("✅ Googleスプレッドシート認証完了 & シート取得完了")

# B1セルから target_site を取得
target_site = worksheet_slot.acell("B1").value.strip()
if not target_site:
    raise ValueError("スプレッドシートB1セルに target_site のURLが設定されていません")
print(f"✅ target_site 取得: {target_site}")

# ========= シート読み取り =========
rows = worksheet_slot.get('A4:G')

# 0時〜9時未満は'2'、それ以外は'1'
target_flag = '2' if (0 <= now.hour < 9) else '1'
print(f"[{datetime.now():%Y-%m-%d %H:%M:%S}] Current Time: {now}, Target Flag: {target_flag}")

# G列(個別設定)がtarget_flagに一致する行を抽出
filtered = [r for r in rows if len(r) >= 7 and r[6].strip() == target_flag]
print(f"[{datetime.now():%Y-%m-%d %H:%M:%S}] Filtered rows: {len(filtered)} 件")

# 台番号（B列）
filtered_dai_numbers = [r[1].strip() for r in filtered]

# URL（F列）
filtered_urls = [r[5].strip() if len(r) >= 6 else "" for r in filtered]

# ========= 数値パーサ =========

def to_int_or_none(s: str):
    """
    文字列から整数を抽出。
    - '▲230' や 全角マイナス '－230' / '−230' も負数として解釈
    - 桁以外は無視（カンマ、空白、単位など）
    - '1/281' や '１／２８１' / '1/281.4' 形式は '1/' を削除して分母だけを数値化
    - 小数点は無視して整数化（切り捨て）
    """
    if s is None:
        return None
    s = str(s).strip()
    # 全角数字・全角スラッシュ→半角
    trans_map = str.maketrans("０１２３４５６７８９／．", "0123456789/.")
    s_norm = s.translate(trans_map)
    # 先頭が "1/" の場合は削除
    if s_norm.startswith("1/"):
        s = s_norm[2:].lstrip()
    else:
        s = s_norm
    # マイナス記号・負数表現の検出
    negative = False
    if s.startswith(("▲", "-", "－", "−")):
        negative = True
    # 数字または小数点を抽出
    m = re.search(r"(\d+(?:\.\d+)?)", s)
    if not m:
        return None
    # 小数点があれば切り捨て
    try:
        val = math.floor(float(m.group(1)))
    except ValueError:
        return None
    return -val if negative else val

# 本日欄（DBに定義済みのカラムのみを対象）
DB_TODAY_SCHEMA = {
    "確変突入回数": "INTEGER",
    "確変突入率": "INTEGER",
    "継続回数": "INTEGER",
    "継続率": "INTEGER",
    "最終スタート": "INTEGER",
    "最大継続": "INTEGER",
    "最大放出数": "INTEGER",
    "出玉数": "INTEGER",
    "初当り回数": "INTEGER",
    "初当り確率": "INTEGER",
    "大当り過去最高": "INTEGER",
    "大当り回数": "INTEGER",
    "大当り確率": "INTEGER",
    "累計スタート": "INTEGER",
}
def _norm_label(s: str) -> str:
    return re.sub(r"\s+", "", s or "")

DB_TODAY_KEYMAP = {_norm_label(k): k for k in DB_TODAY_SCHEMA.keys()}

def cast_today_value(db_col: str, text: str):
    if text is None:
        return None
    typ = DB_TODAY_SCHEMA.get(db_col, "TEXT")
    if typ == "INTEGER":
        return to_int_or_none(text)
    return text.strip()

# ========= Selenium ヘルパ =========
def dismiss_overlays(driver):
    """クッキー同意などのオーバーレイを可能な範囲で閉じる"""
    texts = ["同意", "同意する", "OK", "閉じる", "許可", "Accept", "I agree", "Close"]
    try:
        candidates = driver.find_elements(By.XPATH, "//button|//a|//div")
        for el in candidates[:80]:
            try:
                label = (el.text or "").strip()
                if any(t in label for t in texts) and el.is_displayed() and el.is_enabled():
                    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
                    try:
                        el.click()
                    except Exception:
                        driver.execute_script("arguments[0].click();", el)
                    time.sleep(0.1)
            except Exception:
                pass
    except Exception:
        pass

def switch_to_frame_containing(driver, by, value, timeout=5):
    """指定ロケータの要素を含む iframe を探索して切替。見つからなければトップのまま"""
    driver.switch_to.default_content()
    end = time.time() + timeout
    while time.time() < end:
        try:
            if driver.find_elements(by, value):
                return True
        except Exception:
            pass
        frames = driver.find_elements(By.TAG_NAME, "iframe")
        for fr in frames:
            try:
                driver.switch_to.default_content()
                driver.switch_to.frame(fr)
                if driver.find_elements(by, value):
                    return True
            except Exception:
                continue
        time.sleep(0.2)
    driver.switch_to.default_content()
    return False

def safe_click(driver, locator, timeout=10):
    """presence→scroll→通常click→JS click の順でフォールバック"""
    el = WebDriverWait(driver, timeout).until(EC.presence_of_element_located(locator))
    try:
        WebDriverWait(driver, timeout).until(EC.element_to_be_clickable(locator))
    except Exception:
        pass
    try:
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
        el.click()
        return
    except Exception:
        pass
    driver.execute_script("arguments[0].click();", el)

def safe_set_value(driver, locator, value, timeout=10):
    """不可視/不可クリックでもJSで値代入（input/change発火）"""
    switch_to_frame_containing(driver, *locator, timeout=timeout)
    el = WebDriverWait(driver, timeout).until(EC.presence_of_element_located(locator))
    try:
        WebDriverWait(driver, timeout).until(EC.visibility_of_element_located(locator))
    except Exception:
        pass
    driver.execute_script("""
        const el = arguments[0], val = arguments[1];
        el.focus();
        const setter = Object.getOwnPropertyDescriptor(HTMLInputElement.prototype, 'value').set;
        setter.call(el, '');
        setter.call(el, val);
        el.dispatchEvent(new Event('input', {bubbles: true}));
        el.dispatchEvent(new Event('change', {bubbles: true}));
    """, el, value)

# ========= スクレイピング設定 =========
batch_size = 30
data_list = []

def open_browser():
    chrome_options = uc.ChromeOptions()
    chrome_options.add_argument("--incognito")
    chrome_options.add_argument("--disk-cache-size=0")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--disable-application-cache")
    chrome_options.add_argument("--disable-cache")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    driver = uc.Chrome(version_main=138, options=chrome_options)  # Chrome 138 を想定
    driver.set_window_size(600, 700)
    return driver

def click_more(driver):
    """「もっと見る」を可能な限り展開"""
    while True:
        try:
            element = WebDriverWait(driver, 7).until(EC.element_to_be_clickable((By.ID, "tblHISTm")))
            if element.is_displayed():
                driver.execute_script("arguments[0].click();", element)
                time.sleep(0.8)
            else:
                break
        except TimeoutException:
            break
        except Exception as e:
            print(f"「もっと見る」エラー: {e}")
            break

# ====== LazyLoad & フルページスクショ用ヘルパ ======
CLICK_MORE_SELECTORS = [
    "#tblHISTm",
    ".load-more",
    "button[aria-label='もっと見る']",
]
def click_possible_buttons(driver):
    """「もっと見る」系のボタンを可能な範囲で開く"""
    for sel in CLICK_MORE_SELECTORS:
        try:
            btns = driver.find_elements(By.CSS_SELECTOR, sel)
        except Exception:
            btns = []
        for b in btns[:5]:
            try:
                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", b)
                time.sleep(0.1)
                driver.execute_script("arguments[0].click();", b)
                time.sleep(0.3)
            except Exception:
                pass

def force_eager_images(driver):
    """lazy画像を可能な限り即時ロードへ"""
    driver.execute_script("""
        (function(){
            const imgs = document.querySelectorAll('img');
            for (const img of imgs) {
                try {
                    if ('loading' in img) img.loading = 'eager';
                    const lazy = img.getAttribute('data-src') || img.getAttribute('data-lazy-src');
                    if (lazy && (!img.src || img.src.trim() === '')) img.src = lazy;
                } catch(e){}
            }
        })();
    """)

def auto_scroll_to_bottom(driver, max_time=45, step=900, pause=0.4, idle_wait=1.2):
    """
    LazyLoad対策：高さが伸びなくなるまでステップスクロール＋ボタン連打＋画像強制読込。
    """
    start = time.time()
    stable_rounds = 0
    prev_h = 0
    while time.time() - start < max_time:
        click_possible_buttons(driver)
        h = driver.execute_script("return Math.max(document.body.scrollHeight, document.documentElement.scrollHeight)")
        if h <= 0:
            break
        y = driver.execute_script("return window.pageYOffset || document.documentElement.scrollTop || document.body.scrollTop || 0")
        while y + step < h:
            y += step
            driver.execute_script(f"window.scrollTo(0, {y});")
            time.sleep(pause)
        driver.execute_script(f"window.scrollTo(0, {h});")
        time.sleep(pause)
        force_eager_images(driver)
        new_h = driver.execute_script("return Math.max(document.body.scrollHeight, document.documentElement.scrollHeight)")
        if new_h <= h + 5 and abs(new_h - prev_h) <= 5:
            stable_rounds += 1
        else:
            stable_rounds = 0
        prev_h = new_h
        if stable_rounds >= 2:
            break
    time.sleep(idle_wait)

def fullpage_screenshot(driver, out_path: str):
    """
    CDPでフルページ1枚に保存（GUI/ヘッドレスなしOK）。
    超縦長(~131072px超)は切れる可能性あり。
    """
    metrics = driver.execute_cdp_cmd("Page.getLayoutMetrics", {})
    content = metrics.get("contentSize", {})
    width  = float(content.get("width", 0.0))  or float(driver.execute_script("return window.innerWidth || 1200"))
    height = float(content.get("height", 0.0)) or float(driver.execute_script("return document.body.scrollHeight || 2000"))
    try:
        dpr = driver.execute_cdp_cmd("Emulation.getMetrics", {}).get("deviceScaleFactor", 1.0)
    except Exception:
        dpr = 1.0
    result = driver.execute_cdp_cmd(
        "Page.captureScreenshot",
        {
            "format": "png",
            "fromSurface": True,
            "captureBeyondViewport": True,
            "clip": {"x": 0.0, "y": 0.0, "width": width, "height": height, "scale": float(dpr) if dpr else 1.0},
        },
    )
    with open(out_path, "wb") as f:
        f.write(base64.b64decode(result["data"]))

# ========= ここから：ループ外のユーティリティ =========
def _collect_labels_from_left_sector(browser):
    """MODE=1用：左のラベル列（td.row-header 配下）を順序通りに取得"""
    base_sel = ("#tblDAbv2 > tr > td > div > table > tbody > tr > "
                "td.row-header > div > table > tbody > tr > td")
    labels = []
    try:
        td_nodes = browser.find_elements(By.CSS_SELECTOR, base_sel)
        if td_nodes:
            inners = td_nodes[0].find_elements(By.CSS_SELECTOR, "div.outer > div.inner")
            for inn in inners:
                txt = (inn.text or "").replace("\xa0", "").strip()
                if txt:
                    labels.append(txt)
    except Exception:
        pass
    return labels

def _extract_today_values_from_block(block_text: str, expected_n: int, label_set: set[str]) -> list[str]:
    """MODE=1用：『本日』セルのテキストから本日ブロックのみを切り出し、ラベル/日前見出しを除去して値だけ返す"""
    lines = [x.strip() for x in (block_text or "").replace("\xa0", "").splitlines() if x.strip()]
    if not lines:
        return []
    # 開始位置（本日）
    try:
        start = next(i for i, x in enumerate(lines) if x == "本日" or x.startswith("本日"))
    except StopIteration:
        return []
    # 次の「○日前」ヘッダで切る
    def is_prevday_header(s: str) -> bool:
        return bool(re.fullmatch(r"\d+日前", s)) or ("日前" in s)
    end = None
    for i in range(start + 1, len(lines)):
        if is_prevday_header(lines[i]):
            end = i
            break
    segment = lines[start + 1: end] if end is not None else lines[start + 1:]
    # ラベル・日付見出しを除外して値だけに
    cleaned = [s for s in segment if (s != "本日") and ("日前" not in s) and (s not in label_set)]
    # 期待数に揃える（超過切捨て）
    if expected_n > 0:
        cleaned = cleaned[:expected_n]
    return cleaned

# ========= MODE=2: tbody 直取り =========
def _find_in_any_frame(driver, xpath, timeout=8):
    """全iframeを横断して xpath の最初の一致要素を返す（見つけたframeにswitchした状態）"""
    end = time.time() + timeout
    visited = set()
    while time.time() < end:
        try:
            driver.switch_to.default_content()
            els = driver.find_elements(By.XPATH, xpath)
            if els:
                return els[0]
            frames = driver.find_elements(By.TAG_NAME, "iframe")
            for fr in frames:
                if fr.id in visited:
                    continue
                visited.add(fr.id)
                try:
                    driver.switch_to.default_content()
                    driver.switch_to.frame(fr)
                    els = driver.find_elements(By.XPATH, xpath)
                    if els:
                        return els[0]
                except Exception:
                    continue
        except Exception:
            pass
        time.sleep(0.2)
    raise TimeoutException(f"要素が見つかりませんでした: {xpath}")

def collect_today_by_tbody(driver):
    """
    MODE=2: <tbody id^="tblDAb"> を特定し、各 tr の
      td[1] = ラベル, td[2] = 本日値
    を順に抽出して返す。
    戻り値: (labels, values)
    """
    tbody = _find_in_any_frame(driver, "//tbody[starts-with(@id,'tblDAb')]", timeout=10)
    rows = tbody.find_elements(By.XPATH, "./tr")
    labels, values = [], []

    def norm(s: str) -> str:
        return (s or "").replace("\u00a0", " ").replace("\r", "\n").replace("\n", " ").strip()

    for tr in rows:
        tds = tr.find_elements(By.TAG_NAME, "td")
        if len(tds) < 2:
            continue
        lab = norm(tds[0].text) or norm(tds[0].get_attribute("textContent") or "")
        val = norm(tds[1].text) or norm(tds[1].get_attribute("textContent") or "")
        if lab or val:
            labels.append(lab)
            values.append(val)

    m = min(len(labels), len(values))
    return labels[:m], values[:m]

# ========= MODE=3: ul.nc-border-a > li .title/.value =========
def _collect_pairs_in_current_context(driver):
    """
    現在の frame(もしくはトップ) から:
      ul.nc-border-a > li ... の各行に対し .title / .value を DOM順で抽出
    空欄でも ["", ""] としてペアを追加してズレ防止。
    複数の UL がある場合はそれぞれをグループとして返す。
    戻り値: groups = [ [ [title, value], ... ], ... ]
    """
    groups = []
    uls = driver.find_elements(By.CSS_SELECTOR, "ul.nc-border-a")
    for ul in uls:
        lis = ul.find_elements(By.TAG_NAME, "li")
        items = []
        for li in lis:
            try:
                title_els = li.find_elements(By.CSS_SELECTOR, ".title")
                value_els = li.find_elements(By.CSS_SELECTOR, ".value")
                title = (title_els[0].text if title_els else "").replace("\u00a0", " ").strip()
                value = (value_els[0].text if value_els else "").replace("\u00a0", " ").strip()
                items.append([title, value])  # 空でも保持
            except Exception:
                items.append(["", ""])
        if items:
            groups.append(items)
    return groups

def collect_title_value_pairs(browser, max_iframe_depth=4):
    """
    MODE=3: ul.nc-border-a > li から .title/.value ペアを DOM 順に取得
    複数ULがある場合、既知ラベル一致数 × 行数 で最良ULを選択
    戻り値: (labels, values)
    """
    expected_norm = set(DB_TODAY_KEYMAP.keys())

    def score_group(g):
        titles = [re.sub(r"\s+", "", t or "") for (t, _) in g]
        matches = sum(1 for t in titles if t in expected_norm)
        return (matches, len(g))

    best = None
    best_score = (-1, -1)

    # まずトップ
    try:
        groups = _collect_pairs_in_current_context(browser)
        for g in groups:
            sc = score_group(g)
            if sc > best_score:
                best, best_score = g, sc
    except Exception:
        pass

    # iframe 再帰探索
    def dfs(depth):
        nonlocal best, best_score
        if depth > max_iframe_depth:
            return
        frames = browser.find_elements(By.TAG_NAME, "iframe")
        for i in range(len(frames)):
            try:
                browser.switch_to.frame(i)
                groups = _collect_pairs_in_current_context(browser)
                for g in groups:
                    sc = score_group(g)
                    if sc > best_score:
                        best, best_score = g, sc
                dfs(depth + 1)
            except Exception:
                pass
            finally:
                browser.switch_to.parent_frame()
    dfs(1)
    browser.switch_to.default_content()

    if not best:
        raise RuntimeError("ul.nc-border-a の title/value ペアが見つかりませんでした")

    labels = [t for (t, _) in best]
    values = [v for (_, v) in best]
    return labels, values

# ========= 共通：モード分岐ラッパ =========
def _pad_to(vals: list[str], n: int) -> list[str]:
    if len(vals) < n:
        return vals + [""] * (n - len(vals))
    return vals[:n]

def collect_today_by_column_fallback(browser) -> tuple[list[str], list[str]]:
    """MODE=1: 列インデックス追跡 + 列コンテナ + ブロック解析のフォールバック"""
    label_lines = _collect_labels_from_left_sector(browser)
    labels = label_lines[:]
    expected_n = len(labels)
    today_vals: list[str] = []
    try:
        header_node = browser.find_element(By.XPATH, "//*[normalize-space(text())='本日']")
        vals1 = browser.execute_script("""
            const node = arguments[0], expectedN = arguments[1];
            function norm(s){
                if (s == null) return "";
                const t = String(s).replace(/\\u00a0/g," ").replace(/\\r?\\n/g," ");
                return t.replace(/\\s{2,}/g," ").trim();
            }
            let cell = node.closest('td,th') || node.closest('[role="cell"],[role="columnheader"]');
            if (!cell) return {mode:"no-cell"};
            let row = cell.closest('tr');
            let table = row ? row.closest('table') : null;
            if (!row || !table) return {mode:"no-table"};
            const colIndex = (cell.cellIndex != null ? cell.cellIndex : Array.from(row.children).indexOf(cell));
            const startRow = row.rowIndex + 1;
            const results = [];
            for (let i = startRow; i < table.rows.length && results.length < expectedN; i++) {
                const r = table.rows[i];
                const td = r.cells[colIndex];
                if (!td) { results.push(""); continue; }
                results.push(norm(td.textContent));
            }
            while (results.length < expectedN) results.push("");
            if (results.length > expectedN) results.length = expectedN;
            return {mode:"table", results};
        """, header_node, expected_n)

        if isinstance(vals1, dict) and vals1.get("mode") == "table":
            today_vals = [v if isinstance(v, str) else "" for v in vals1["results"]]
        else:
            today_vals = []

        if not any(v for v in today_vals):
            col_td = None
            try:
                col_td = header_node.find_element(By.XPATH, ".//ancestor::*[contains(@class,'column')][1]")
            except Exception:
                try:
                    col_td = header_node.find_element(By.XPATH, ".//ancestor::td[1]")
                except Exception:
                    col_td = None

            vals2: list[str] = []
            if col_td is not None:
                nodes = col_td.find_elements(
                    By.CSS_SELECTOR,
                    "div.inner.nc-text-align-right, [class*='nc-text-align-right'], [data-value], [aria-label]"
                )
                for n in nodes:
                    txt = (n.get_attribute("textContent") or "").strip()
                    if not txt:
                        txt = (n.get_attribute("data-value") or "").strip()
                    if not txt:
                        txt = (n.get_attribute("aria-label") or "").strip()
                    vals2.append(txt.replace("\u00a0", " ").replace("\r\n", " ").replace("\n", " ").strip())
                today_vals = _pad_to(vals2, expected_n)

        if not today_vals:
            try:
                header_cell = browser.find_element(By.XPATH, "//td[contains(normalize-space(),'本日')] | //th[contains(normalize-space(),'本日')]")
                block = (header_cell.text or "")
            except Exception:
                block = ""
            tmp = _extract_today_values_from_block(block, expected_n=expected_n, label_set=set(labels))
            today_vals = _pad_to(tmp, expected_n)
    except Exception:
        today_vals = [""] * expected_n

    today_vals = _pad_to(today_vals, expected_n)
    return labels, today_vals

def collect_today(browser) -> tuple[list[str], list[str]]:
    """グローバル TODAY_MODE(1/2/3) に従って (labels, values) を返す"""
    if TODAY_MODE == 1:
        labels, vals = collect_today_by_column_fallback(browser)
        vals = _pad_to(vals, len(labels))
    elif TODAY_MODE == 2:
        labels, vals = collect_today_by_tbody(browser)
        vals = _pad_to(vals, len(labels))
    elif TODAY_MODE == 3:
        labels, vals = collect_title_value_pairs(browser)
        vals = _pad_to(vals, len(labels))
    else:
        raise ValueError(f"Unsupported TODAY_MODE: {TODAY_MODE}")

    labels = [("" if x is None else str(x)) for x in labels]
    vals   = [("" if x is None else str(x)) for x in vals]
    return labels, vals

# ========= SKU 用ユーティリティ =========
def get_starting_sku_seq(db_path: str, date_prefix: str) -> int:
    """
    既存DBに date_prefix(YYYYMMDD) で始まる SKU があれば、その最大末尾4桁+1 を返す。
    なければ 1 を返す。SKU は TEXT 前提（例: 20250812####）
    """
    try:
        with sqlite3.connect(db_path) as conn:
            cur = conn.cursor()
            # 末尾4桁を整数化して最大値をとる（SKUは12桁: 8+4）
            cur.execute(f"""
                SELECT MAX(CAST(SUBSTR(SKU, 9, 4) AS INTEGER))
                FROM {table_name}
                WHERE SKU LIKE ? AND LENGTH(SKU) = 12
            """, (f"{date_prefix}%",))
            max_suffix = cur.fetchone()[0]
            return (int(max_suffix) + 1) if max_suffix is not None else 1
    except Exception as e:
        print(f"SKU初期シーケンス取得エラー: {e}（1から開始）")
        return 1

def load_cookies(browser):
    try:
        with open(COOKIE_FILE, "r") as file:
            cookies = json.load(file)
            for cookie in cookies:
                if "sameSite" in cookie:
                    cookie.pop("sameSite")
                browser.add_cookie(cookie)
        print("Cookies loaded.")
    except FileNotFoundError:
        print("No cookies found.")

# ========= メインループ =========
# 当日(=前日)のSKUシーケンス開始番号を取得
sku_seq = get_starting_sku_seq(db_path, sku_date)

total = len(filtered_dai_numbers)
for batch_start in range(0, total, batch_size):
    batch_end = min(batch_start + batch_size, total)

    browser = open_browser()
    time.sleep(1.0)
    browser.get(target_site)
    load_cookies(browser)
    browser.refresh()
    time.sleep(1.0)

    for index in range(batch_start, batch_end):
        dai_number = filtered_dai_numbers[index]
        url = filtered_urls[index]

        # ★ DBカラム名で保持（基礎 + 本日値）
        data_entry = {
            "台番号": dai_number,     # 取得失敗時の保険
            "pscubeURL": url,
            "取得更新日": None,
            "機種名": None,
            "svgデータ": None,
            "SKU": f"{sku_date}{sku_seq:04d}",   # 追加: 各行にSKU採番
            "実行日": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }
        sku_seq += 1  # 次行に備えてインクリメント

        # 本日カラムを None 初期化（欠損時に古い値が残らない）
        for col in DB_TODAY_SCHEMA.keys():
            data_entry[col] = None

        try:
            print(f"\nProcessing 台番号(検索値): {dai_number} (Index: {index+1})")
            dismiss_overlays(browser)

            # 「台番号で探す」→入力→検索
            safe_click(browser, (By.XPATH, "//div[contains(@class, 'search-item') and contains(., '台番号で探す')]"), timeout=12)
            safe_set_value(browser, (By.NAME, "cd_dai"), dai_number, timeout=12)

            # 検索ボタン
            WebDriverWait(browser, 2).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, ".nc-da-search-btn.nc-da-search-btn-submit"))
            )
            btns = browser.find_elements(By.CSS_SELECTOR, ".nc-da-search-btn.nc-da-search-btn-submit")
            target_btn = next((b for b in btns if b.is_displayed() and b.is_enabled()), btns[0] if btns else None)
            if not target_btn:
                raise Exception("検索ボタンが見つかりませんでした。")
            try:
                browser.execute_script("arguments[0].scrollIntoView({block:'center'});", target_btn)
                target_btn.click()
            except Exception:
                browser.execute_script("arguments[0].click();", target_btn)

            time.sleep(7)
            click_more(browser)

            # 取得更新日
            try:
                div_element = browser.find_element(By.ID, "upYMDhms")
                update_date = div_element.text.replace(" 更新", "").strip()
                update_date_obj = datetime.strptime(update_date, "%Y/%m/%d %H:%M")
                data_entry["取得更新日"] = update_date_obj.strftime("%Y-%m-%d %H:%M:%S")
                print(f"取得更新日: {data_entry['取得更新日']}")
            except Exception as e:
                print(f"取得更新日 取得失敗: {e}")

            # 台番号
            try:
                h2_element = browser.find_element(By.CSS_SELECTOR, "h2.nc-text-align-left")
                v = h2_element.text.replace("台番号 ", "").strip()
                data_entry["台番号"] = v or data_entry["台番号"]
            except Exception:
                print("台番号 取得失敗（検索値を保持）")

            # 機種名
            try:
                title = browser.find_element(By.ID, "divKI-name")
                data_entry["機種名"] = title.text.strip()
            except Exception:
                print("機種名 取得失敗")

            # --- 本日欄：3モード分岐（返り値は (labels, values) に統一） ---
            try:
                print(f"=== 本日欄処理開始 (MODE={TODAY_MODE}) ===")
                labels, today_vals = collect_today(browser)

                saved = 0
                for lab, val in zip(labels, today_vals):
                    key = DB_TODAY_KEYMAP.get(_norm_label(lab))
                    if not key:
                        continue
                    data_entry[key] = cast_today_value(key, (None if val == "" else val))
                    saved += 1

                if saved == 0:
                    print("[WARN] 本日欄：既知ラベルに一致する保存件数が0件でした（ラベル変更の可能性）")

                print(f"[本日] rows={len(labels)} / 保存件数={saved}")
                print("=== 本日欄処理終了 ===\n")
            except Exception as e:
                print(f"本日テーブル取得エラー: {e}")

            # 履歴（テーブル最上段=1回前）— ヘッダ名＝DB名でそのまま格納
            try:
                hist_table_element = browser.find_element(By.ID, "tblHIST")
                hist_rows = hist_table_element.find_elements(By.TAG_NAME, "tr")
                if len(hist_rows) <= 1:
                    print("履歴データなし")
                else:
                    header_cells = hist_rows[0].find_elements(By.TAG_NAME, "th")
                    if not header_cells:
                        header_cells = hist_rows[0].find_elements(By.TAG_NAME, "td")
                    header_texts = [(c.text or "").strip() for c in header_cells]
                    wanted = ["時刻", "スタート", "ステータス", "出玉pt"]
                    idx = {name: next((i for i, t in enumerate(header_texts) if name in t), None) for name in wanted}

                    body_rows = hist_rows[1:]
                    max_n = min(100, len(body_rows))
                    for n in range(1, max_n + 1):
                        row = body_rows[n - 1]
                        cells = row.find_elements(By.TAG_NAME, "td")
                        def tx(i):
                            return (cells[i].text or "").strip() if (i is not None and i < len(cells)) else ""
                        data_entry[f"時刻{n}回前"]       = tx(idx["時刻"]) or None
                        data_entry[f"スタート{n}回前"]   = to_int_or_none(tx(idx["スタート"]))
                        data_entry[f"ステータス{n}回前"] = tx(idx["ステータス"]) or None
                        data_entry[f"出玉pt{n}回前"]     = (to_int_or_none(tx(idx["出玉pt"])) if idx["出玉pt"] is not None else None)

                    print(f"履歴行数(上→下): {len(body_rows)} → 保存: {max_n} 行（最上段=1回前）")
            except Exception as e:
                print(f"履歴取得エラー: {e}")

            # svgデータ
            try:
                svg_elements = browser.find_elements(By.CSS_SELECTOR, 'svg[version="1.1"]')
                if len(svg_elements) >= 2:
                    data_entry["svgデータ"] = browser.execute_script('return arguments[0].outerHTML;', svg_elements[1])
                    print("svgデータ取得 OK")
                else:
                    print("svgデータ 要素不足")
            except Exception as e:
                print(f"svgデータ取得エラー: {e}")

            # ★★★ LazyLoad 後にフルページスクショ ★★★
            try:
                auto_scroll_to_bottom(browser, max_time=45, step=900, pause=0.4, idle_wait=1.2)
                png_path = os.path.join(base_path, f"{today}_{dai_number}.png")
                fullpage_screenshot(browser, png_path)
                print(f"[SHOT] saved: {png_path}")
            except Exception as e:
                print(f"[SHOT] スクショ失敗: {e}")

        except Exception as e:
            print(f"台番号 {dai_number} 処理中エラー: {e}")

        data_list.append(data_entry)

    try:
        browser.quit()
    except Exception as e:
        print(f"ブラウザ終了エラー: {e}")

    # ★ 閉じてから次を開くまで待機
    time.sleep(10)   # ← ここで待機秒数を調整

# ========= DataFrame へ =========
df = pd.DataFrame(data_list)
print(f"収集完了: {len(df)} 件")

df.to_csv(csv_path, index=False, encoding="utf-8-sig")
df.to_excel(excel_path, index=False)

# ========= DB 書き込み（動的 UPSERT） =========
# ★ SKU を BASE_COLS に追加
BASE_COLS = ["pscubeURL", "取得更新日", "機種名", "svgデータ", "SKU", "実行日"] + list(DB_TODAY_SCHEMA.keys())
HIST_RE = re.compile(r"^(時刻|スタート|ステータス|出玉pt)(\d+)回前$")

def ensure_exec_unique_schema(conn: sqlite3.Connection):
    cur = conn.cursor()
    # 台番号 + 実行日 の重複を禁止
    cur.execute("""
        CREATE UNIQUE INDEX IF NOT EXISTS ux_result_table_exec
        ON result_table(台番号, 実行日)
    """)
    conn.commit()


def upsert_result(conn: sqlite3.Connection, row: dict) -> int:
    machine_no = (row.get("台番号") or "").strip()
    exec_ts    = (row.get("実行日") or "").strip()  # ← 実行日をキーとして使う
    if not machine_no or not exec_ts:
        print("台番号 or 実行日 なしのためスキップ")
        return 0

    # 動的履歴カラム（1..100）
    hist_cols = []
    for k in row.keys():
        m = HIST_RE.match(k)
        if m:
            n = int(m.group(2))
            if 1 <= n <= 100:
                hist_cols.append(k)
    hist_cols.sort(key=lambda c: int(HIST_RE.match(c).group(2)))

    cur = conn.cursor()

    # 既存判定：台番号 + 実行日
    cur.execute(
        f"SELECT 1 FROM {table_name} WHERE 台番号 = ? AND 実行日 = ?",
        (machine_no, exec_ts)
    )
    exists = (cur.fetchone() is not None)

    # UPDATE 時は「実行日」を上書きしない（キーだから）
    update_cols = [c for c in BASE_COLS if c != "実行日"] + hist_cols
    set_cols    = [f"{c} = ?" for c in update_cols]
    params_update = [row.get(c) for c in update_cols] + [machine_no, exec_ts]

    if exists:
        cur.execute(
            f"UPDATE {table_name} SET {', '.join(set_cols)} WHERE 台番号 = ? AND 実行日 = ?",
            params_update,
        )
        return cur.rowcount
    else:
        insert_cols = ["pscubeURL", "取得更新日", "台番号", "機種名", "svgデータ", "SKU", "実行日"] \
                      + list(DB_TODAY_SCHEMA.keys()) + hist_cols
        insert_vals = [
            row.get("pscubeURL"),
            row.get("取得更新日"),
            machine_no,
            row.get("機種名"),
            row.get("svgデータ"),
            row.get("SKU"),
            exec_ts,
        ] + [row.get(c) for c in DB_TODAY_SCHEMA.keys()] + [row.get(c) for c in hist_cols]
        placeholders = ", ".join(["?"] * len(insert_vals))
        cur.execute(
            f"INSERT INTO {table_name} ({', '.join(insert_cols)}) VALUES ({placeholders})",
            insert_vals,
        )
        return cur.rowcount

updated, inserted = 0, 0
with sqlite3.connect(db_path) as conn:
    # （任意）重複をDBレベルで防ぐ
    ensure_exec_unique_schema(conn)  # ← ①を入れた場合は呼ぶ

    for r in data_list:
        machine_no = (r.get("台番号") or "").strip()
        exec_ts    = (r.get("実行日") or "").strip()
        before_exists = False
        if machine_no and exec_ts:
            before_exists = conn.execute(
                f"SELECT 1 FROM {table_name} WHERE 台番号 = ? AND 実行日 = ?",
                (machine_no, exec_ts)
            ).fetchone() is not None

        rc = upsert_result(conn, r)
        if rc:
            if before_exists:
                updated += 1
            else:
                inserted += 1
    conn.commit()

print(f"DB 書き込み完了: 挿入 {inserted} / 更新 {updated}")

# ======== STEP2: PNG → WebP 変換 ========
def save_webp_from_png(png_path, dst_dir):
    filename  = os.path.basename(png_path).replace(".png", ".webp")
    save_path = os.path.join(dst_dir, filename)
    try:
        with Image.open(png_path) as img:
            img.save(save_path, "WEBP", quality=80)
        print(f"[INFO] WebP保存完了: {save_path}")
    except Exception as e:
        print(f"[ERROR] WebP保存失敗 {filename}: {e}")

def convert_all_png_to_webp(src_dir, dst_dir):
    png_files = [f for f in os.listdir(src_dir) if f.lower().endswith(".png")]
    print(f"[INFO] PNGファイル {len(png_files)} 件をWebPに変換")
    for fname in png_files:
        save_webp_from_png(os.path.join(src_dir, fname), dst_dir)
    print("[INFO] 全PNGのWebP変換完了")

convert_all_png_to_webp(base_path, webp_dir)
print(f"✅ STEP2 完了: 出力 {webp_dir}")

# ======================
# 終了
# ======================
end_time = time.time()
print(f"[INFO] スクリプト完了（実行時間: {end_time - start_time:.2f} 秒）")


✅ Googleスプレッドシート認証完了 & シート取得完了
✅ target_site 取得: https://www.pscube.jp/dedamajyoho-P-townDMMpachi/c709505
[2025-08-18 20:30:40] Current Time: 2025-08-18 20:30:38.311762, Target Flag: 1
[2025-08-18 20:30:40] Filtered rows: 47 件
Cookies loaded.

Processing 台番号(検索値): 301 (Index: 1)
取得更新日: 2025-08-18 20:22:00
=== 本日欄処理開始 (MODE=3) ===
[本日] rows=13 / 保存件数=13
=== 本日欄処理終了 ===

履歴行数(上→下): 2 → 保存: 2 行（最上段=1回前）
svgデータ取得 OK
[SHOT] saved: C:\Users\stray\myenv310\ootake-maruhachi-p\pic\20250817\20250817_301.png

Processing 台番号(検索値): 302 (Index: 2)
取得更新日: 2025-08-18 20:22:00
=== 本日欄処理開始 (MODE=3) ===
[本日] rows=13 / 保存件数=13
=== 本日欄処理終了 ===

履歴データなし
svgデータ取得 OK
[SHOT] saved: C:\Users\stray\myenv310\ootake-maruhachi-p\pic\20250817\20250817_302.png

Processing 台番号(検索値): 303 (Index: 3)
取得更新日: 2025-08-18 20:22:00
=== 本日欄処理開始 (MODE=3) ===
[本日] rows=13 / 保存件数=13
=== 本日欄処理終了 ===

履歴データなし
svgデータ取得 OK
[SHOT] saved: C:\Users\stray\myenv310\ootake-maruhachi-p\pic\20250817\20250817_303.png

Processing 台番号(検索値): 304

KeyboardInterrupt: 